# ART SLIDE — 全作品の詳しい解説を取得し直す

現在GitHubに保存されている**全作品**を対象に、画像・作品ID・ゲームの記録を一切消さず、解説だけを補強するノートブック。

- **シカゴ美術館**：公式APIの `description`（長い公式解説）を優先し、足りない場合は `short_description` と作品目録（技法・制作年・寸法・来歴など）を合わせる
- **The Met**：公式APIの作品メタデータを取得し、確認できた事実をまとめる。美術館による長い解説が公式APIから取得できない場合、存在するかのように書かない
- **既存の長文を短い文に置き換えない**：更新文が十分長い場合のみ採用
- **権利情報を記録**：シカゴ美術館の公式 `description` の利用条件はCC BY 4.0。出典・加工内容を保存
- **GitHubに直接保存**：`src/games/art-slide/data/description-overrides.ts` に作品IDごとの説明をまとめて保存。処理結果を `description-report.json` に保存
- **中断に備えたチェックポイント**：同じColabランタイムで再実行した場合は取得済みの作品を再利用。GitHub上の新規作品も再確認

## 実行手順

1. Colabの鍵アイコン（Secrets）に `GITHUB_TOKEN` を登録し、ノートブックへのアクセスを許可。トークンには `ryouy/slides` のContents「Read and write」権限が必要
2. 「ランタイム」→「すべてのセルを実行」
3. 完了後に表示されるコミットURLと `description-report.json` で、実際の更新数・文字数を確認

**追加の画像ダウンロードはしません。削除もしません。** 既存の `artworks.ts`、`additional-artworks.ts`、デイリー用固定リストは一切編集しません。

## 詳細解説の翻訳（任意）

標準では、公式の詳しい英語原文を取得し、各言語の短い概要と併記する。日本語などの長い翻訳が必要なら、コードの `TRANSLATE_LANGUAGES = ["ja"]` などを設定し、Colab Secretsに `DEEPL_API_KEY` を登録できる。DeepLの利用には別途料金・文字数制限が生じる場合がある。翻訳しない場合、他言語では長い原文を英語で表示する。

In [ ]:
%pip -q install requests beautifulsoup4


In [ ]:
# ART SLIDE — refresh detailed descriptions for every artwork already in GitHub.
# No image downloads, no deletes, and no edits to the original catalogue.
REPO = "ryouy/slides"
BRANCH = "main"
MAX_CHARS = 7500
API_PAUSE_SECONDS = 0.38
SAVE_CHECKPOINT_EVERY = 15
CHECKPOINT_PATH = "/content/art_slide_description_checkpoint.json"
DRY_RUN = False
# Optional DeepL translations: only enabled when you explicitly add target languages.
# Supported values: "ja", "zh", "ko", "es", "fr", "de". Requires DEEPL_API_KEY in Colab Secrets.
TRANSLATE_LANGUAGES = []

import base64, collections, datetime, html, json, os, re, statistics, time
from urllib.parse import urlencode
import requests
from bs4 import BeautifulSoup
from google.colab import userdata

ROOT = "https://api.github.com/repos/" + REPO
AIC = "https://api.artic.edu/api/v1/artworks/"
MET = "https://collectionapi.metmuseum.org/public/collection/v1/objects/"
CC_BY = "https://creativecommons.org/licenses/by/4.0/"
CATALOG_BASE = "src/games/art-slide/data/"
OVERLAY_PATH = CATALOG_BASE + "description-overrides.ts"
REPORT_PATH = CATALOG_BASE + "description-report.json"

token = (userdata.get("GITHUB_TOKEN") or "").strip()
if not token: raise RuntimeError("Add GITHUB_TOKEN to Colab Secrets, then enable notebook access.")
gh_session = requests.Session()
gh_session.headers.update({
    "Authorization": "Bearer " + token,
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
    "User-Agent": "art-slide-description-enricher/1.0"
})
museum = requests.Session()
museum.headers.update({
    "User-Agent": "art-slide-description-enricher/1.0",
    "AIC-User-Agent": "art-slide-description-enricher/1.0",
})

def response_json(session, method, url, **kwargs):
    for attempt in range(6):
        try:
            r = session.request(method, url, timeout=(15, 80), **kwargs)
            if r.status_code in (429, 500, 502, 503, 504) or (
                r.status_code == 403 and r.headers.get("Retry-After")):
                time.sleep(min(65, int(r.headers.get("Retry-After") or (3 * 2**attempt))))
                continue
            if r.status_code == 404: return None
            if not r.ok:
                message = r.text[:250]
                raise RuntimeError(f"{r.status_code} {url}: {message}")
            return r.json()
        except requests.RequestException:
            if attempt == 5: raise
            time.sleep(min(60, 3 * 2**attempt))
    raise RuntimeError("Too many retry attempts: " + url)

def gh(method, path, payload=None):
    return response_json(gh_session, method, ROOT + path, json=payload)

def current_ref():
    return gh("GET", "/git/ref/heads/" + BRANCH)["object"]["sha"]

def git_file(path, revision, optional=False):
    j = response_json(gh_session, "GET", ROOT + "/contents/" + path,
                      params={"ref": revision})
    if j is None:
        if optional: return ""
        raise RuntimeError("Required repository file missing: " + path)
    content = j.get("content")
    if content:
        return base64.b64decode(content).decode("utf-8")
    # Content API omits contents for large files. Fall back to git blob API.
    if j.get("sha"):
        blob = gh("GET", "/git/blobs/" + j["sha"])
        if blob and blob.get("encoding") == "base64":
            return base64.b64decode(blob["content"]).decode("utf-8")
    raise RuntimeError("Cannot read " + path + ". Refusing an unsafe catalogue update.")

def parse_catalogue(source, var):
    pattern = r"export const " + var + r": Artwork\[\]\s*=\s*(\[[\s\S]*?\]);"
    m = re.search(pattern, source)
    if not m: raise ValueError("Failed to parse " + var)
    return json.loads(m.group(1))

def parse_overrides(source):
    if not source.strip(): return {}
    pattern = r"export const DESCRIPTION_OVERRIDES: Record<string, Partial<Artwork>> = (\{[\s\S]*?\});"
    m = re.search(pattern, source)
    if not m: raise ValueError("Unexpected description-overrides.ts format. No changes published.")
    return json.loads(m.group(1))

def read_snapshot(sha):
    base = parse_catalogue(git_file(CATALOG_BASE + "artworks.ts", sha), "ARTWORKS")
    extra = parse_catalogue(git_file(CATALOG_BASE + "additional-artworks.ts", sha), "NEW_ARTWORKS")
    overrides = parse_overrides(git_file(OVERLAY_PATH, sha, optional=True))
    records = {a["id"]: a for a in base + extra}
    if len(records) != len(base) + len(extra):
        raise RuntimeError("Duplicate artwork IDs in catalogue. Refusing to publish.")
    return records, overrides

def clean(s):
    if not s: return ""
    soup = BeautifulSoup(html.unescape(str(s)), "html.parser")
    for el in soup(["script", "style"]): el.decompose()
    txt = soup.get_text(" ", strip=True)
    return re.sub(r"\s+", " ", txt).strip()

def short(s, maxlen):
    s = clean(s)
    if len(s) <= maxlen: return s
    return s[:maxlen].rsplit(" ", 1)[0].rstrip(" ,;") + "…"

def split_sentences(s):
    return re.split(r"(?<=[.!?])\s+(?=[A-Z0-9])", s)

def add_fact(facts, label, raw, maxlen=240):
    value = short(raw, maxlen)
    if value and value.lower() not in ("unknown", "n/a", "none", "not dated"):
        facts.append(label + ": " + value + ".")
    
def chicago_catalogue(a, work):
    facts = []
    add_fact(facts, "Recorded artist or maker", a.get("artist_display") or work["artist"], 250)
    add_fact(facts, "Date", a.get("date_display") or work["year"])
    add_fact(facts, "Materials and technique", a.get("medium_display") or work.get("medium"))
    add_fact(facts, "Artistic style or school", a.get("style_title"))
    add_fact(facts, "Place of creation", a.get("place_of_origin"))
    add_fact(facts, "Classification", a.get("classification_title") or a.get("artwork_type_title"))
    add_fact(facts, "Physical dimensions", a.get("dimensions"), 270)
    tags = a.get("subject_titles") or []
    if tags: add_fact(facts, "Museum catalogue subjects", ", ".join(str(t) for t in tags[:7]))
    add_fact(facts, "Museum credit line", a.get("credit_line"), 280)
    # Provenance and exhibition records are separate cataloguing facts, not curator interpretation.
    provenance = short(a.get("provenance_text"), 380)
    if provenance: add_fact(facts, "Recorded ownership history (excerpt)", provenance, 400)
    exhibition = short(a.get("exhibition_history"), 280)
    if exhibition: add_fact(facts, "Exhibition record (excerpt)", exhibition, 300)
    return " ".join(facts)

def met_catalogue(a, work):
    facts = []
    add_fact(facts, "Artist", a.get("artistDisplayName") or work["artist"])
    add_fact(facts, "Artist background", a.get("artistDisplayBio"), 250)
    add_fact(facts, "Date", a.get("objectDate") or work["year"])
    add_fact(facts, "Materials and technique", a.get("medium") or work.get("medium"))
    add_fact(facts, "Classification", a.get("classification") or a.get("objectName"))
    add_fact(facts, "Culture", a.get("culture"))
    add_fact(facts, "Period", a.get("period"))
    add_fact(facts, "Place of origin", ", ".join(filter(None, [a.get("country"), a.get("region"), a.get("city")])))
    add_fact(facts, "Physical dimensions", a.get("dimensions"), 270)
    add_fact(facts, "Department", a.get("department"))
    tags = [t.get("term") for t in a.get("tags") or [] if isinstance(t, dict) and t.get("term")]
    if tags: add_fact(facts, "Catalogue subjects", ", ".join(tags[:7]))
    add_fact(facts, "Acquisition credit", a.get("creditLine"), 280)
    return " ".join(facts)

def fetch_detail(work):
    museum_name, url = str(work.get("museum", "")), str(work.get("sourceUrl", ""))
    m = re.search(r"artic\.edu/artworks/(\d+)", url)
    if m:
        time.sleep(API_PAUSE_SECONDS)
        fields = ",".join([
            "id", "title", "artist_display", "date_display", "medium_display",
            "style_title", "place_of_origin", "classification_title", "artwork_type_title",
            "subject_titles", "dimensions", "credit_line", "provenance_text",
            "exhibition_history", "description", "short_description"
        ])
        j = response_json(museum, "GET", AIC + m.group(1), params={"fields": fields})
        if not j or not j.get("data"): return None
        a = j["data"]
        long_form = clean(a.get("description"))
        short_form = clean(a.get("short_description"))
        official = long_form or short_form
        facts = chicago_catalogue(a, work)
        prose = official or (
            f"{work['title']} is a work attributed in the Art Institute of Chicago "
            f"catalogue to {work['artist']}."
        )
        composed = prose + ("\n\nCatalogue information\n" + facts if facts else "")
        if len(composed) > MAX_CHARS:
            composed = short(composed, MAX_CHARS)
        kind = "museum_text" if official else "metadata_summary"
        lic = {
            "descriptionSourceUrl": url,
            "descriptionAttribution": "Art Institute of Chicago",
            "descriptionLicense": "CC BY 4.0",
            "descriptionLicenseUrl": CC_BY,
            "descriptionModifications": "Museum text converted to plain text; catalogue facts added; abridged if necessary.",
        } if official else {}
        return {"description": composed, "descriptionKind": kind, **lic}, "AIC " + ("museum prose" if official else "metadata only")
    m = re.search(r"metmuseum\.org/art/collection/search/(\d+)", url)
    if m:
        time.sleep(API_PAUSE_SECONDS)
        j = response_json(museum, "GET", MET + m.group(1))
        if not j: return None
        facts = met_catalogue(j, work)
        opening = (f"{work['title']} is recorded in the Metropolitan Museum of Art "
                   f"collection as a work by {work['artist']}.")
        composed = opening + ("\n\nCatalogue information\n" + facts if facts else "")
        if len(composed) > MAX_CHARS: composed = short(composed, MAX_CHARS)
        # The Met public collection API has catalogue fields, but does not
        # guarantee a licensed, full curatorial essay for every object.
        return {"description": composed, "descriptionKind": "metadata_summary"}, "Met metadata only"
    return None

def optional_translations(text):
    langs = [x for x in TRANSLATE_LANGUAGES if x in {"ja", "zh", "ko", "es", "fr", "de"}]
    if not langs: return {}
    key = (userdata.get("DEEPL_API_KEY") or "").strip()
    if not key:
        print("DEEPL_API_KEY not set. Skipping optional translations.")
        return {}
    endpoint = ("https://api-free.deepl.com/v2/translate"
                if key.endswith(":fx") else "https://api.deepl.com/v2/translate")
    out = {}
    for lang in langs:
        try:
            r = requests.post(endpoint,
                headers={"Authorization": "DeepL-Auth-Key " + key},
                data={"text": text, "source_lang": "EN", "target_lang": lang.upper()},
                timeout=(15, 90))
            if r.status_code in (429, 456):
                print("Translation limit reached. Keeping original English:", r.status_code)
                break
            r.raise_for_status()
            translated = r.json()["translations"][0]["text"].strip()
            if translated: out[lang] = translated
            time.sleep(0.3)
        except Exception as exc:
            print("Translation unavailable for", lang, ":", type(exc).__name__)
    return out

# Resumable within the same Colab runtime; no access token is written to the checkpoint.
progress = {}
if os.path.exists(CHECKPOINT_PATH):
    try:
        with open(CHECKPOINT_PATH, "r", encoding="utf-8") as handle:
            progress = json.load(handle)
        if not isinstance(progress, dict): progress = {}
        print("Recovered", len(progress), "completed artworks from this Colab runtime.")
    except Exception:
        progress = {}
errors = {}
processed = set()
successful_commit = False

for scan_pass in range(3):
    ref = current_ref()
    all_artworks, previous_overrides = read_snapshot(ref)
    print(f"Pass {scan_pass + 1}: repository {ref[:12]} has {len(all_artworks)} paintings.")
    print("Descriptions already in the app are preserved unless longer verified content is available.")
    for ix, (id, work) in enumerate(all_artworks.items(), 1):
        if id in processed: continue
        if id in progress and isinstance(progress[id], dict):
            processed.add(id)
            continue
        current = {**work, **previous_overrides.get(id, {})}
        try:
            fetched = fetch_detail(current)
            if not fetched:
                errors[id] = "Museum detail unavailable; original text retained"
                progress[id] = {}
            else:
                patch, source_note = fetched
                prev_text = str(current.get("description") or "")
                candidate = str(patch.get("description") or "")
                # Never replace a more informative description with something shorter.
                if len(candidate) < len(prev_text) + 25:
                    progress[id] = {}
                else:
                    translated = optional_translations(candidate)
                    if translated:
                        patch["descriptionTranslations"] = {
                            **(current.get("descriptionTranslations") or {}), **translated
                        }
                    progress[id] = patch
        except Exception as exc:
            errors[id] = type(exc).__name__ + ": " + str(exc)[:130]
            progress[id] = {}
        processed.add(id)
        if ix % 10 == 0 or ix == len(all_artworks):
            done = len(processed)
            print(f"Scanned {done}/{len(all_artworks)}; descriptions improved:",
                  sum(bool(x.get("description")) for x in progress.values()),
                  "API errors:", len(errors))
        if ix % SAVE_CHECKPOINT_EVERY == 0:
            with open(CHECKPOINT_PATH, "w", encoding="utf-8") as f:
                json.dump(progress, f, ensure_ascii=False)
    with open(CHECKPOINT_PATH, "w", encoding="utf-8") as f:
        json.dump(progress, f, ensure_ascii=False)

    latest_ref = current_ref()
    if latest_ref != ref:
        print("GitHub changed while we were fetching. Scanning the latest catalogue again.")
        # Existing improvements remain cached; newly added works will be fetched.
        continue

    updated_overrides = dict(previous_overrides)
    count_improved = 0
    for id in all_artworks:
        patch = progress.get(id)
        if patch and patch.get("description"):
            updated_overrides[id] = {**updated_overrides.get(id, {}), **patch}
            count_improved += 1

    effective = [{**a, **updated_overrides.get(i, {})} for i, a in all_artworks.items()]
    length_list = sorted(len(w.get("description") or "") for w in effective)
    quality = collections.Counter(w.get("descriptionKind") or "original-brief" for w in effective)
    report = {
        "generatedAt": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "repository": REPO, "baseCommit": ref, "totalArtworks": len(all_artworks),
        "updatedDescriptionsThisRun": count_improved,
        "descriptionTypes": dict(quality),
        "medianLength": statistics.median(length_list),
        "minLength": min(length_list),
        "maxLength": max(length_list),
        "under200": sum(n < 200 for n in length_list),
        "under400": sum(n < 400 for n in length_list),
        "over600": sum(n >= 600 for n in length_list),
        "failedRequests": errors,
        "notes": "Only official APIs were used; missing curator prose was replaced only with factual catalogue summaries.",
    }
    override_source = (
        "import type { Artwork } from '../types';\n\n"
        "/** Museum-API details, updated by notebooks/ART_SLIDE_description_refresh.ipynb. */\n"
        "export const DESCRIPTION_OVERRIDES: Record<string, Partial<Artwork>> = "
        + json.dumps(updated_overrides, ensure_ascii=False, indent=2) + ";\n"
    )
    print("New coverage:", json.dumps({k:v for k,v in report.items()
        if k in ("totalArtworks", "updatedDescriptionsThisRun", "descriptionTypes",
                 "medianLength", "under200", "under400", "over600")}, ensure_ascii=False))
    print("API errors:", len(errors), "See", REPORT_PATH, "after publishing.")
    if DRY_RUN:
        print("DRY_RUN=True: no GitHub files were changed.")
        successful_commit = True
        break
    # Single atomic commit. Images and daily-puzzle pools are untouched.
    parent_tree = gh("GET", "/git/commits/" + ref)["tree"]["sha"]
    tree_items = []
    for filepath, payload in {
        OVERLAY_PATH: override_source,
        REPORT_PATH: json.dumps(report, ensure_ascii=False, indent=2) + "\n",
    }.items():
        blob = gh("POST", "/git/blobs", {
            "content": base64.b64encode(payload.encode("utf-8")).decode("ascii"),
            "encoding": "base64",
        })
        tree_items.append({"path": filepath, "mode": "100644",
                           "type": "blob", "sha": blob["sha"]})
    tree_sha = gh("POST", "/git/trees", {
        "base_tree": parent_tree, "tree": tree_items
    })["sha"]
    new_commit = gh("POST", "/git/commits", {
        "message": "Enrich detailed descriptions for " + str(len(all_artworks)) + " paintings",
        "tree": tree_sha, "parents": [ref],
    })["sha"]
    try:
        gh("PATCH", "/git/refs/heads/" + BRANCH, {
            "sha": new_commit, "force": False
        })
    except RuntimeError as exc:
        if "409" in str(exc) or "422" in str(exc):
            print("GitHub changed during publish. Rechecking, without force-pushing.")
            continue
        raise
    successful_commit = True
    print("Published successfully:")
    print("https://github.com/" + REPO + "/commit/" + new_commit)
    print("Total artworks:", len(all_artworks))
    print("New/extended descriptions:", count_improved)
    print("No image, game state, catalogue or daily-pool files were modified.")
    break

if not successful_commit:
    print("The branch was changing repeatedly. Current results are saved to", CHECKPOINT_PATH)
    print("Run this cell again to refresh and publish without losing completed API calls.")


## 実行時の確認事項

404作品は**このノートブック作成時点**のGitHub登録数です。実行時に最新の作品数を読み直すため、その後作品が追加されていても対象に含まれます。

公式APIに美術館執筆の解説がない作品は、作者・年・技法・テーマ・サイズ・来歴など、取得できた事実だけで説明を作る。すべての作品に同じ分量の長文を保証するものではない。APIエラーで取得できなかった作品は既存の説明を維持し、レポートに記録する。

GitHubのブランチが変更されていれば最新状態を読み直して安全に再試行する。画像を消したり、作品一覧を初期化する必要はない。